## Running the models

For each of the deformation types—viscous, elastic, and plastic, we will run the corresponding ASPECT parameter file.

<div class="alert alert-block alert-info">
    <b>Note:</b> If you are running this notebook on the hubzero platform, make sure to uncomment the following lines to enable the notebook to run aspect.
</div>

In [ ]:
#import hublib.use
#%use aspect-3.1.0


In [1]:
%%capture
! aspect-release ./prm_files/elastic.prm


-----------------------------------------------------------------------------
--                             This is ASPECT                              --
-- The Advanced Solver for Planetary Evolution, Convection, and Tectonics. --
-----------------------------------------------------------------------------
--     . version 3.2.0-pre (main, 7242f0bef)
--     . using deal.II 9.7.0
--     .       with 32 bit indices
--     .       with vectorization level 1 (SSE2, 2 doubles, 128 bits)
--     . using Trilinos 16.2.0
--     . using p4est 2.8.7
--     . using Geodynamic World Builder 1.1.1
--     . running in OPTIMIZED mode
--     . running with 1 MPI process
-----------------------------------------------------------------------------

-----------------------------------------------------------------------------
-- For information on how to cite ASPECT, see:
--   https://aspect.geodynamics.org/citing.html?ver=3.2.0-pre&dg=1&mf=1&sha=7242f0bef&src=code
-----------------------------------

OSError: [Errno 5] Input/output error

In [ ]:
%%capture
! aspect-release ./prm_files/viscous.prm


In [ ]:
%%capture
! aspect-release ./prm_files/plastic.prm


> In this above cell, it is assumed that the ASPECT directory is installed system-wide or that you are using ASPECT in the HubZero environment with notebooks. If not, modify the ASPECT executable to the location where it is installed.

> ASPECT can be configured in either "release" mode or "debug" mode. Release mode is more optimized than debug mode, and will usually run much faster but lacks debugging symbols and more descriptive error messages. If you are making changes to ASPECT or altering parameter files, it might be a good idea to try running in debug mode in order to resolve any problems. The above code is calling the release version using `aspect-release` . If you want to try running debug mode, call `aspect-debug` instead.

In [2]:
import os
os.environ['VTK_DEFAULT_OPENGL_WINDOW'] = 'vtkEGLRenderWindow' # Suppresses a warning with Pyvista
# Load the relevant libraries
import numpy as np
import pyvista as pv
import glob
from IPython.display import HTML


### Plotting the solution field

ASPECT outputs the solution fields in binary, .vtu, format. We will use the python package `pyvista` to read the output files and `matplotlib` to plot the solution fields.

>The mesh in all three cases is structured, we don't have to worry about interpolating the solution to a structured grid for plotting.

In [3]:
def plot_deformation(output_dir, field):
    '''
    Process a series of VTU files in the given output directory, returning the 'field'
    data for plotting.
    Inputs:
    -----------
    output_dir : str
        Path to the folder containing solution VTU files (expected in solution/solution-*.vtu)
    field : str
        Name of the field we want to look at in the solution
    label : str
        Label for the field being plotted
    '''

    solution_file_names = sorted(glob.glob(output_dir + 'solution/solution-*.pvtu'))

    plotter = pv.Plotter(off_screen=True)
    mesh = pv.read(solution_file_names[0])

    # Create the first frame
    plotter.add_mesh(mesh, scalars=field, show_scalar_bar=False)
    plotter.add_axes()

    # We know that the model is XY plane so the following
    # lines just sets up the camera
    plotter.view_xy()
    plotter.camera.roll = 0
    plotter.camera.parallel_projection = True

    gif_file = 'images/' + output_dir.split('/')[0] + '.gif'
    plotter.open_gif(gif_file, fps=4)

    # Update function for each frame
    def update(frame):
        frame_int = int(frame)
        new_mesh = pv.read(solution_file_names[frame_int])
        plotter.add_mesh(new_mesh, scalars=field, cmap='RdBu', \
                         scalar_bar_args={"vertical": True, "title": "Slab"})
        plotter.add_text(f"Time step: {frame}", name="tstep")
        plotter.add_text(output_dir.split('/')[0],  position='upper_right',
                        font_size=14)

    n_frames = len(solution_file_names)

    for i in range(n_frames):
        update(i)
        plotter.write_frame()

    plotter.close()


In [ ]:
plot_deformation('elastic_deformation/', 'slab')
plot_deformation('viscous_deformation/', 'slab')
plot_deformation('plastic_deformation/', 'slab')


IndexError: list index out of range

: 

Now, you would have the gifs saved in the current directory. This cell visualizes the model outputs side by side for comparison.

<img src="images/elastic_deformation.gif" width="500" align="center">

<br><br>

<img src="images/viscous_deformation.gif" width="500" align="center">

<br><br>

<img src="images/plastic_deformation.gif" width="500" align="center">

<br><br>


Notice, the different deformation styles of the slab under similar external forces. The elastic slab deforms and rebounds to its original shape, the viscous slab flows, and the plastic slab stretches internally and form shear bands. Both viscous and plastic deformation are permanent.

 &nbsp;<div style="text-align: right">  
    &rarr; <b>NEXT: [Stokes flow benchmark](./7_stokes_flow_benchmark.ipynb) </b> <a href=""></a> &nbsp;&nbsp;
     <img src="../../../assets/education-gem-notebooks_icon.png" alt="icon" width="4%">
  </div>